# 05 — Final Data Export & BI Optimization

**Sector:** Aviation / Transportation  
**Project:** US Flight Ops Dashboard (Capstone 2)  
**Objective:** Consolidate external lookup tables and final cleaned datasets into a denormalized structure optimized for Tableau.

---

## 5.1 Setup

In [12]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Auto-detect processed data path
def find_processed_path():
    candidates = [
        '../data/processed/',
        'data/processed/',
        './'
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, 'flights_cleaned.csv')):
            print(f"Data found at: {os.path.abspath(path)}")
            return path
    raise FileNotFoundError("Could not find flights_cleaned.csv. Run 02_cleaning.ipynb first.")

PROCESSED_PATH = find_processed_path()
TABLEAU_PATH = os.path.join(PROCESSED_PATH, 'tableau/')
os.makedirs(TABLEAU_PATH, exist_ok=True)

print("Loading cleaned flights data...")
df = pd.read_csv(f'{PROCESSED_PATH}flights_cleaned.csv', low_memory=False)
df['DATE'] = pd.to_datetime(df['DATE'])

df_nc = df[df['CANCELLED'] == 0].copy()

print(f"Loaded {len(df):,} total flights ({len(df_nc):,} non-cancelled)")

Data found at: /Users/yashraghubanshi/Downloads/dva/data/processed
Loading cleaned flights data...
Loaded 100,000 total flights (98,446 non-cancelled)


## 5.2 Dataset 1 — Airline Summary

In [13]:
# Airline-level KPIs
airline_summary = df.groupby(['AIRLINE', 'AIRLINE_NAME']).agg(
    total_flights=('FLIGHT_NUMBER', 'count'),
    cancelled_flights=('CANCELLED', 'sum'),
    diverted_flights=('DIVERTED', 'sum'),
    avg_departure_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arrival_delay=('ARRIVAL_DELAY', 'mean'),
    median_arrival_delay=('ARRIVAL_DELAY', 'median'),
    total_delay_minutes=('ARRIVAL_DELAY', lambda x: x[x > 0].sum()),
    avg_distance=('DISTANCE', 'mean'),
    avg_air_system_delay=('AIR_SYSTEM_DELAY', 'mean'),
    avg_airline_delay=('AIRLINE_DELAY', 'mean'),
    avg_weather_delay=('WEATHER_DELAY', 'mean'),
    avg_late_aircraft_delay=('LATE_AIRCRAFT_DELAY', 'mean'),
    avg_security_delay=('SECURITY_DELAY', 'mean')
).reset_index()

# Computed KPIs
airline_summary['cancellation_rate_pct'] = (airline_summary['cancelled_flights'] / airline_summary['total_flights'] * 100).round(2)
airline_summary['on_time_flights'] = airline_summary['total_flights'] - airline_summary['cancelled_flights']

# On-time rate (need to compute from non-cancelled)
on_time_rates = df_nc.groupby('AIRLINE').apply(lambda x: (x['ARRIVAL_DELAY'] <= 15).mean() * 100).reset_index()
on_time_rates.columns = ['AIRLINE', 'on_time_rate_pct']
airline_summary = airline_summary.merge(on_time_rates, on='AIRLINE')

airline_summary = airline_summary.round(2)

output_file = f'{TABLEAU_PATH}tableau_airline_summary.csv'
airline_summary.to_csv(output_file, index=False)
print(f"Airline summary: {airline_summary.shape} -> {output_file}")
airline_summary.head()

Airline summary: (14, 18) -> ../data/processed/tableau/tableau_airline_summary.csv


,AIRLINE,AIRLINE_NAME,total_flights,cancelled_flights,diverted_flights,avg_departure_delay,avg_arrival_delay,median_arrival_delay,total_delay_minutes,avg_distance,avg_air_system_delay,avg_airline_delay,avg_weather_delay,avg_late_aircraft_delay,avg_security_delay,cancellation_rate_pct,on_time_flights,on_time_rate_pct
0,AA,American Airlines Inc.,12612,195,38,9.55,4.16,-6.0,157296.0,1040.43,2.48,4.15,0.64,4.10,0.04,1.55,12417,81.52
1,AS,Alaska Airlines Inc.,2958,10,7,1.69,-1.09,-6.0,21662.0,1184.36,1.63,2.49,0.08,1.93,0.02,0.34,2948,87.89
2,B6,JetBlue Airways,4480,75,9,11.43,6.98,-5.0,66005.0,1080.01,3.84,3.93,0.30,5.52,0.06,1.67,4405,77.98
3,DL,Delta Air Lines Inc.,14921,64,28,7.18,-0.08,-8.0,133565.0,853.62,2.10,3.16,0.57,2.19,0.00,0.43,14857,87.11
4,EV,Atlantic Southeast Airlines,9853,298,29,8.40,6.16,-4.0,125107.0,465.20,2.96,3.75,0.33,4.54,0.00,3.02,9555,80.97


## 5.3 Dataset 2 — Airport Summary

In [14]:
# Airport-level metrics (origin airports)
airport_summary = df_nc.groupby(['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 
                                  'ORIGIN_STATE', 'ORIGIN_LAT', 'ORIGIN_LONG']).agg(
    total_departures=('FLIGHT_NUMBER', 'count'),
    avg_dep_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arr_delay=('ARRIVAL_DELAY', 'mean'),
    median_dep_delay=('DEPARTURE_DELAY', 'median'),
    delay_rate=('IS_DELAYED', 'mean'),
    avg_taxi_out=('TAXI_OUT', 'mean'),
    unique_airlines=('AIRLINE', 'nunique'),
    unique_destinations=('DESTINATION_AIRPORT', 'nunique')
).reset_index()

airport_summary['delay_rate_pct'] = (airport_summary['delay_rate'] * 100).round(2)
airport_summary['on_time_rate_pct'] = (100 - airport_summary['delay_rate_pct']).round(2)
airport_summary = airport_summary.round(2)

# Also add cancellation data
cancel_by_airport = df.groupby('ORIGIN_AIRPORT')['CANCELLED'].sum().reset_index()
cancel_by_airport.columns = ['ORIGIN_AIRPORT', 'cancelled_flights']
airport_summary = airport_summary.merge(cancel_by_airport, on='ORIGIN_AIRPORT', how='left')

output_file = f'{TABLEAU_PATH}tableau_airport_summary.csv'
airport_summary.to_csv(output_file, index=False)
print(f"Airport summary: {airport_summary.shape} -> {output_file}")
airport_summary.head()

Airport summary: (314, 17) -> ../data/processed/tableau/tableau_airport_summary.csv


,ORIGIN_AIRPORT,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_LAT,ORIGIN_LONG,total_departures,avg_dep_delay,avg_arr_delay,median_dep_delay,delay_rate,avg_taxi_out,unique_airlines,unique_destinations,delay_rate_pct,on_time_rate_pct,cancelled_flights
0,ABE,Lehigh Valley International Airport,Allentown,PA,40.65,-75.44,39,18.72,14.54,-1.0,0.15,12.59,2,3,15.38,84.62,2
1,ABI,Abilene Regional Airport,Abilene,TX,32.41,-99.68,37,14.27,10.70,-1.0,0.24,8.43,1,1,24.32,75.68,3
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,35.04,-106.61,340,7.26,3.50,-2.0,0.15,12.14,10,23,15.04,84.96,5
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,45.45,-98.42,9,-0.33,-3.67,-3.0,0.11,15.22,1,1,11.11,88.89,0
4,ABY,Southwest Georgia Regional Airport,Albany,GA,31.54,-84.19,15,-2.93,-3.73,-5.0,0.07,12.00,1,1,6.67,93.33,0


## 5.4 Dataset 3 — Monthly Trends

In [15]:
# Monthly trends (for time-series dashboard)
monthly_trends = df.groupby(['MONTH', 'MONTH_NAME']).agg(
    total_flights=('FLIGHT_NUMBER', 'count'),
    cancelled_flights=('CANCELLED', 'sum'),
    diverted_flights=('DIVERTED', 'sum'),
    avg_departure_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arrival_delay=('ARRIVAL_DELAY', 'mean'),
    total_delay_minutes=('ARRIVAL_DELAY', lambda x: x[x > 0].sum()),
    avg_air_system_delay=('AIR_SYSTEM_DELAY', 'mean'),
    avg_airline_delay=('AIRLINE_DELAY', 'mean'),
    avg_weather_delay=('WEATHER_DELAY', 'mean'),
    avg_late_aircraft_delay=('LATE_AIRCRAFT_DELAY', 'mean')
).reset_index()

monthly_trends['cancellation_rate_pct'] = (monthly_trends['cancelled_flights'] / monthly_trends['total_flights'] * 100).round(2)

# On-time rate per month
monthly_on_time = df_nc.groupby('MONTH').apply(lambda x: (x['ARRIVAL_DELAY'] <= 15).mean() * 100).reset_index()
monthly_on_time.columns = ['MONTH', 'on_time_rate_pct']
monthly_trends = monthly_trends.merge(monthly_on_time, on='MONTH')

monthly_trends = monthly_trends.round(2)

output_file = f'{TABLEAU_PATH}tableau_monthly_trends.csv'
monthly_trends.to_csv(output_file, index=False)
print(f"Monthly trends: {monthly_trends.shape} -> {output_file}")
monthly_trends

Monthly trends: (12, 14) -> ../data/processed/tableau/tableau_monthly_trends.csv


,MONTH,MONTH_NAME,total_flights,cancelled_flights,diverted_flights,avg_departure_delay,avg_arrival_delay,total_delay_minutes,avg_air_system_delay,avg_airline_delay,avg_weather_delay,avg_late_aircraft_delay,cancellation_rate_pct,on_time_rate_pct
0,1,January,8102,220,16,9.45,5.29,100244.0,2.67,3.30,0.48,4.73,2.72,79.59
1,2,February,7440,330,17,11.70,8.28,106593.0,3.10,3.98,0.86,5.13,4.44,77.52
2,3,March,8581,193,20,9.54,4.81,101497.0,2.54,3.42,0.41,4.22,2.25,80.81
3,4,April,8391,75,23,7.77,3.22,89778.0,2.44,2.77,0.40,3.87,0.89,83.02
4,5,May,8644,100,33,9.20,4.09,101734.0,2.26,3.48,0.66,4.25,1.16,82.65
5,6,June,8684,157,34,13.40,9.01,131643.0,3.14,4.21,0.73,5.91,1.81,77.14
6,7,July,8927,79,25,12.31,7.22,126607.0,2.74,4.47,0.47,5.46,0.88,78.61
7,8,August,8630,89,26,9.79,4.72,104431.0,2.57,3.54,0.45,4.46,1.03,81.68
8,9,September,8058,33,5,5.47,-0.14,69129.0,1.83,2.96,0.34,2.50,0.41,87.23
9,10,October,8203,43,20,5.18,-0.36,66941.0,1.63,2.54,0.32,2.64,0.52,87.89


## 5.5 Dataset 4 — Route Analysis

In [16]:
# Route-level analysis (top 200 routes by volume)
route_analysis = df_nc.groupby(['ROUTE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
                                 'ORIGIN_CITY', 'DEST_CITY',
                                 'ORIGIN_STATE', 'DEST_STATE']).agg(
    flight_count=('FLIGHT_NUMBER', 'count'),
    avg_arr_delay=('ARRIVAL_DELAY', 'mean'),
    median_arr_delay=('ARRIVAL_DELAY', 'median'),
    delay_rate=('IS_DELAYED', 'mean'),
    avg_distance=('DISTANCE', 'mean'),
    unique_airlines=('AIRLINE', 'nunique')
).reset_index()

route_analysis['delay_rate_pct'] = (route_analysis['delay_rate'] * 100).round(2)

# Keep top 200 routes
top_routes = route_analysis.nlargest(200, 'flight_count')
top_routes = top_routes.round(2)

output_file = f'{TABLEAU_PATH}tableau_route_analysis.csv'
top_routes.to_csv(output_file, index=False)
print(f"Route analysis (top 200): {top_routes.shape} -> {output_file}")
top_routes.head(10)

Route analysis (top 200): (200, 14) -> ../data/processed/tableau/tableau_route_analysis.csv


,ROUTE,ORIGIN_AIRPORT,DESTINATION_AIRPORT,ORIGIN_CITY,DEST_CITY,ORIGIN_STATE,DEST_STATE,flight_count,avg_arr_delay,median_arr_delay,delay_rate,avg_distance,unique_airlines,delay_rate_pct
2257,LAX -> SFO,LAX,SFO,Los Angeles,San Francisco,CA,CA,237,16.41,0.0,0.29,337.0,6,28.69
3797,SFO -> LAX,SFO,LAX,San Francisco,Los Angeles,CA,CA,217,17.67,3.0,0.32,337.0,6,32.26
2224,LAX -> JFK,LAX,JFK,Los Angeles,New York,CA,NY,203,5.92,-5.0,0.21,2475.0,5,21.18
2050,JFK -> LAX,JFK,LAX,New York,Los Angeles,NY,CA,202,-4.70,-8.0,0.16,2475.0,5,16.00
2150,LAS -> LAX,LAS,LAX,Las Vegas,Los Angeles,NV,CA,183,12.01,2.0,0.25,236.0,7,25.14
2074,JFK -> SFO,JFK,SFO,New York,San Francisco,NY,CA,170,-3.99,-12.0,0.18,2586.0,5,17.65
3085,ORD -> LGA,ORD,LGA,Chicago,New York,IL,NY,159,9.13,-8.0,0.27,733.0,4,26.75
188,ATL -> MCO,ATL,MCO,Atlanta,Orlando,GA,FL,155,7.54,-3.0,0.17,404.0,4,16.77
2332,LGA -> ORD,LGA,ORD,New York,Chicago,NY,IL,147,5.20,-9.0,0.21,733.0,3,21.09
1689,HNL -> OGG,HNL,OGG,Honolulu,Kahului,HI,HI,144,2.67,0.0,0.06,100.0,1,6.25


## 5.6 Dataset 5 — Delay Causes Breakdown

In [17]:
# Delay cause breakdown by airline and month
delay_causes = df_nc.groupby(['AIRLINE', 'AIRLINE_NAME', 'MONTH', 'MONTH_NAME']).agg(
    total_flights=('FLIGHT_NUMBER', 'count'),
    avg_air_system_delay=('AIR_SYSTEM_DELAY', 'mean'),
    avg_security_delay=('SECURITY_DELAY', 'mean'),
    avg_airline_delay=('AIRLINE_DELAY', 'mean'),
    avg_late_aircraft_delay=('LATE_AIRCRAFT_DELAY', 'mean'),
    avg_weather_delay=('WEATHER_DELAY', 'mean'),
    total_air_system_delay=('AIR_SYSTEM_DELAY', 'sum'),
    total_airline_delay=('AIRLINE_DELAY', 'sum'),
    total_late_aircraft_delay=('LATE_AIRCRAFT_DELAY', 'sum'),
    total_weather_delay=('WEATHER_DELAY', 'sum'),
    total_security_delay=('SECURITY_DELAY', 'sum')
).reset_index()

delay_causes = delay_causes.round(2)

output_file = f'{TABLEAU_PATH}tableau_delay_causes.csv'
delay_causes.to_csv(output_file, index=False)
print(f"Delay causes: {delay_causes.shape} -> {output_file}")
delay_causes.head()

Delay causes: (162, 15) -> ../data/processed/tableau/tableau_delay_causes.csv


,AIRLINE,AIRLINE_NAME,MONTH,MONTH_NAME,total_flights,avg_air_system_delay,avg_security_delay,avg_airline_delay,avg_late_aircraft_delay,avg_weather_delay,total_air_system_delay,total_airline_delay,total_late_aircraft_delay,total_weather_delay,total_security_delay
0,AA,American Airlines Inc.,1,January,720,2.47,0.00,3.57,6.30,0.67,1779.0,2570.0,4535.0,481.0,0.0
1,AA,American Airlines Inc.,2,February,623,4.02,0.26,4.35,4.29,1.71,2504.0,2712.0,2670.0,1063.0,161.0
2,AA,American Airlines Inc.,3,March,785,2.93,0.12,3.96,3.85,0.27,2298.0,3110.0,3019.0,214.0,93.0
3,AA,American Airlines Inc.,4,April,798,3.16,0.00,3.59,5.22,0.51,2518.0,2864.0,4165.0,410.0,0.0
4,AA,American Airlines Inc.,5,May,753,2.50,0.00,3.56,5.03,2.31,1886.0,2678.0,3787.0,1742.0,0.0


## 5.7 Dataset 6 — Hourly Departure Patterns

In [18]:
# Hourly patterns (by day of week and hour)
hourly_pattern = df_nc.groupby(['DAY_OF_WEEK', 'DAY_NAME', 'DEP_HOUR', 'TIME_OF_DAY']).agg(
    flight_count=('FLIGHT_NUMBER', 'count'),
    avg_dep_delay=('DEPARTURE_DELAY', 'mean'),
    avg_arr_delay=('ARRIVAL_DELAY', 'mean'),
    delay_rate=('IS_DELAYED', 'mean')
).reset_index()

hourly_pattern['delay_rate_pct'] = (hourly_pattern['delay_rate'] * 100).round(2)
hourly_pattern = hourly_pattern.round(2)

output_file = f'{TABLEAU_PATH}tableau_hourly_pattern.csv'
hourly_pattern.to_csv(output_file, index=False)
print(f"Hourly patterns: {hourly_pattern.shape} -> {output_file}")
hourly_pattern.head(10)

Hourly patterns: (164, 9) -> ../data/processed/tableau/tableau_hourly_pattern.csv


,DAY_OF_WEEK,DAY_NAME,DEP_HOUR,TIME_OF_DAY,flight_count,avg_dep_delay,avg_arr_delay,delay_rate,delay_rate_pct
0,1,Monday,0,Night,37,22.00,20.27,0.32,32.43
1,1,Monday,1,Night,13,3.31,-1.15,0.15,15.38
2,1,Monday,2,Night,3,-6.00,-1.67,0.33,33.33
3,1,Monday,4,Night,1,-5.00,-15.00,0.00,0.00
4,1,Monday,5,Morning,330,7.87,2.85,0.10,9.76
5,1,Monday,6,Morning,1053,3.19,-1.78,0.08,7.63
6,1,Monday,7,Morning,1002,3.78,-0.64,0.12,11.62
7,1,Monday,8,Morning,913,6.65,1.33,0.13,12.83
8,1,Monday,9,Morning,905,8.06,4.08,0.15,15.30
9,1,Monday,10,Morning,913,7.18,2.24,0.15,15.13


## 5.8 Dataset 7 — Flight-Level Sample (for Tableau drill-down)

In [19]:
# Stratified sample for flight-level Tableau analysis
# 5% sample to keep file manageable for Tableau Public
np.random.seed(42)
sample_cols = ['DATE', 'MONTH', 'MONTH_NAME', 'DAY_OF_WEEK', 'DAY_NAME', 'QUARTER',
               'AIRLINE', 'AIRLINE_NAME', 'FLIGHT_NUMBER',
               'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'ROUTE',
               'ORIGIN_CITY', 'ORIGIN_STATE', 'DEST_CITY', 'DEST_STATE',
               'ORIGIN_LAT', 'ORIGIN_LONG', 'DEST_LAT', 'DEST_LONG',
               'SCHEDULED_DEPARTURE_TIME', 'DEP_HOUR', 'TIME_OF_DAY',
               'DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'DISTANCE', 'DISTANCE_CATEGORY',
               'ELAPSED_TIME', 'AIR_TIME', 'TAXI_OUT', 'TAXI_IN',
               'CANCELLED', 'DIVERTED', 'CANCELLATION_REASON_DESC',
               'DELAY_CATEGORY', 'IS_DELAYED', 'PRIMARY_DELAY_CAUSE',
               'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
               'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

# Only keep columns that exist
available_cols = [c for c in sample_cols if c in df.columns]

df_tableau_sample = df[available_cols].sample(frac=0.05, random_state=42)

output_file = f'{TABLEAU_PATH}tableau_flights_sample.csv'
df_tableau_sample.to_csv(output_file, index=False)
file_size = os.path.getsize(output_file) / (1024**2)
print(f"Flights sample: {df_tableau_sample.shape} -> {output_file} ({file_size:.2f} MB)")
df_tableau_sample.head()

Flights sample: (5000, 42) -> ../data/processed/tableau/tableau_flights_sample.csv (1.38 MB)


,DATE,MONTH,MONTH_NAME,DAY_OF_WEEK,DAY_NAME,QUARTER,AIRLINE,AIRLINE_NAME,FLIGHT_NUMBER,ORIGIN_AIRPORT,...,DIVERTED,CANCELLATION_REASON_DESC,DELAY_CATEGORY,IS_DELAYED,PRIMARY_DELAY_CAUSE,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
75721,2015-10-26,10,October,1,Monday,Q4,AA,American Airlines Inc.,2050,11057,...,0,Not Cancelled,Moderate Delay (16-60 min),1.0,Air System,18.0,0.0,0.0,0.0,0.0
80184,2015-05-19,5,May,2,Tuesday,Q2,OO,Skywest Airlines Inc.,4827,FNT,...,0,Not Cancelled,On Time / Early,0.0,No Delay,0.0,0.0,0.0,0.0,0.0
19864,2015-08-17,8,August,1,Monday,Q3,AS,Alaska Airlines Inc.,73,SIT,...,0,Not Cancelled,On Time / Early,0.0,No Delay,0.0,0.0,0.0,0.0,0.0
76699,2015-07-05,7,July,7,Sunday,Q3,AA,American Airlines Inc.,801,CLT,...,0,Not Cancelled,On Time / Early,0.0,No Delay,0.0,0.0,0.0,0.0,0.0
92991,2015-10-31,10,October,6,Saturday,Q4,OO,Skywest Airlines Inc.,6343,10800,...,0,Not Cancelled,Moderate Delay (16-60 min),1.0,Late Aircraft,0.0,0.0,0.0,46.0,0.0


## 5.9 Export Manifest

In [20]:
# List all exported files
print("=" * 70)
print("FINAL LOAD PREPARATION — EXPORT MANIFEST")
print("=" * 70)
print(f"\nOutput directory: {os.path.abspath(TABLEAU_PATH)}")
print(f"\n{'File':<45} {'Size (MB)':<12} {'Status'}")
print("-" * 70)

total_size = 0
for f in sorted(os.listdir(TABLEAU_PATH)):
    fpath = os.path.join(TABLEAU_PATH, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / (1024**2)
        total_size += size
        print(f"  {f:<43} {size:<10.2f}   Done")

print("-" * 70)
print(f"  {'TOTAL':<43} {total_size:<10.2f}")
print(f"\nAll {len(os.listdir(TABLEAU_PATH))} datasets exported successfully.")
print(f"\nThese files are ready for import into Tableau Public.")
print(f"Upload them as data sources and build your dashboard.")

FINAL LOAD PREPARATION — EXPORT MANIFEST

Output directory: /Users/yashraghubanshi/Downloads/dva/data/processed/tableau

File                                          Size (MB)    Status
----------------------------------------------------------------------
  tableau_airline_summary.csv                 0.00         Done
  tableau_airport_summary.csv                 0.03         Done
  tableau_delay_causes.csv                    0.01         Done
  tableau_flights_sample.csv                  1.38         Done
  tableau_hourly_pattern.csv                  0.01         Done
  tableau_monthly_trends.csv                  0.00         Done
  tableau_route_analysis.csv                  0.01         Done
----------------------------------------------------------------------
  TOTAL                                       1.45      

All 7 datasets exported successfully.

These files are ready for import into Tableau Public.
Upload them as data sources and build your dashboard.


## 5.10 Tableau Dashboard Design Recommendations

### Suggested Dashboard Structure:

**View 1 — Executive Summary:**
- KPI cards: Total Flights, On-Time Rate, Avg Delay, Cancellation Rate
- Monthly trend line with delay overlay
- Airline performance scorecard

**View 2 — Operational Drill-Down:**
- Airport map with delay heat intensity (use lat/long)
- Day x Hour heatmap for delay hotspots
- Delay cause breakdown (stacked bar or treemap)
- Route performance table with conditional formatting

**Interactive Filters:**
- Airline selector
- Month/Quarter filter
- Origin/Destination state filter
- Distance category filter

**Data Source -> File Mapping:**

| Tableau View | Data File |
|---|---|
| KPI Cards | `tableau_airline_summary.csv` |
| Monthly Trends | `tableau_monthly_trends.csv` |
| Airport Map | `tableau_airport_summary.csv` |
| Day x Hour Heatmap | `tableau_hourly_pattern.csv` |
| Delay Causes | `tableau_delay_causes.csv` |
| Route Table | `tableau_route_analysis.csv` |
| Drill-Down | `tableau_flights_sample.csv` |